# OpenAI GPT-Realtime Demo

## 🎙️ Speech-to-Speech AI Assistant with OpenAI's GPT-Realtime API

This notebook demonstrates the new **GPT-Realtime model** using the OpenAI Agents SDK and Gradio for a web interface. The GPT-Realtime model provides:

- 🗣️ **Native speech-to-speech**: Direct audio input to audio output without text intermediary
- ⚡ **Low latency**: Real-time conversation with minimal delay
- 🎯 **Improved quality**: Better natural speech, instruction following, and function calling
- 🌐 **New voices**: Includes Cedar and Marin voices with enhanced naturalness

---

### Key Features Implemented:
- **Real-time voice conversations** using OpenAI's latest `gpt-realtime` model
- **Function calling** for weather queries and other tools
- **Agent handoffs** between specialized agents
- **Interactive Gradio interface** for web deployment
- **Audio streaming** with interruption support

In [ ]:
# Install required dependencies
# Note: Run this cell first if packages are not installed

!pip install --quiet 'openai-agents[voice]' gradio fastapi uvicorn websockets numpy sounddevice pydub

In [ ]:
# Essential imports for GPT-Realtime implementation

import asyncio
import os
import numpy as np
import gradio as gr
import io
import tempfile
import time
from typing import Optional, Tuple, AsyncIterator
from dataclasses import dataclass
from dotenv import load_dotenv

# OpenAI Agents SDK imports for Realtime API
from agents.realtime import RealtimeAgent, RealtimeRunner, RealtimeSession
from agents import function_tool

# Audio processing
try:
    import sounddevice as sd
    from pydub import AudioSegment
    print("✅ Audio libraries loaded successfully")
except ImportError as e:
    print(f"⚠️ Audio library import failed: {e}")
    print("Run: pip install sounddevice pydub")

print("📦 All imports completed successfully!")

In [ ]:
# Environment setup and API key validation

load_dotenv(override=True)

# Check API key
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key and openai_api_key.startswith('sk-proj-') and len(openai_api_key) > 10:
    print(f"✅ OpenAI API key validated (begins with: {openai_api_key[:12]}...)")
else:
    print("❌ OpenAI API key not found or invalid")
    print("Please set OPENAI_API_KEY in your .env file")

# Audio configuration constants
SAMPLE_RATE = 24000  # 24kHz as required by OpenAI Realtime API
CHANNELS = 1  # Mono audio
CHUNK_DURATION = 0.1  # 100ms chunks
FORMAT = np.int16

print(f"🎵 Audio configuration: {SAMPLE_RATE}Hz, {CHANNELS} channel(s), {FORMAT} format")

In [ ]:
# Function tools for the GPT-Realtime agent

@function_tool
def get_weather(city: str) -> str:
    """Get the current weather for a specific city.
    
    Args:
        city: The name of the city to get weather for
        
    Returns:
        A description of the current weather
    """
    # Simulated weather data - in production you'd call a real weather API
    import random
    
    weather_conditions = [
        "sunny and clear",
        "partly cloudy", 
        "overcast with light rain",
        "foggy",
        "snowing lightly"
    ]
    
    temperature = random.randint(-5, 35)  # Celsius
    condition = random.choice(weather_conditions)
    
    return f"The weather in {city} is currently {condition} with a temperature of {temperature}°C."

@function_tool
def get_time() -> str:
    """Get the current time.
    
    Returns:
        The current time as a formatted string
    """
    import datetime
    now = datetime.datetime.now()
    return f"The current time is {now.strftime('%I:%M %p on %B %d, %Y')}."

@function_tool
def calculate(expression: str) -> str:
    """Calculate a mathematical expression safely.
    
    Args:
        expression: A mathematical expression to evaluate (e.g., "2 + 2", "10 * 5")
        
    Returns:
        The result of the calculation
    """
    try:
        # Safe evaluation of basic math expressions
        allowed_chars = set('0123456789+-*/.() ')
        if all(c in allowed_chars for c in expression):
            result = eval(expression)
            return f"The result of {expression} is {result}."
        else:
            return "I can only calculate basic mathematical expressions with numbers and operators."
    except Exception as e:
        return f"I couldn't calculate that expression. Error: {str(e)}"

print("🔧 Function tools defined successfully!")
print("   - get_weather: Get weather information for any city")
print("   - get_time: Get current date and time")
print("   - calculate: Perform basic mathematical calculations")

In [ ]:
# Create GPT-Realtime agents with different specializations

# Main assistant agent with general capabilities
main_agent = RealtimeAgent(
    name="Voice Assistant",
    instructions="""
    You are a helpful voice assistant powered by GPT-Realtime. You can:
    - Have natural conversations with users
    - Answer questions on various topics
    - Help with weather information, time, and calculations
    - Speak in a friendly, conversational tone
    
    Keep your responses concise but helpful. When using tools, explain what you're doing.
    If you can't help with something, politely explain your limitations.
    """,
    tools=[get_weather, get_time, calculate]
)

# Specialized weather agent
weather_agent = RealtimeAgent(
    name="Weather Specialist",
    handoff_description="A weather specialist for detailed weather information and forecasts",
    instructions="""
    You are a weather specialist. You provide detailed weather information and can discuss:
    - Current weather conditions
    - Weather patterns and phenomena
    - Seasonal changes
    - Weather-related advice
    
    Use the get_weather tool when users ask about specific cities.
    Speak with expertise but keep it conversational and easy to understand.
    """,
    tools=[get_weather]
)

# Math helper agent
math_agent = RealtimeAgent(
    name="Math Helper", 
    handoff_description="A math helper for calculations and mathematical questions",
    instructions="""
    You are a helpful math assistant. You can:
    - Perform calculations using the calculate tool
    - Explain mathematical concepts
    - Help solve basic math problems
    - Provide step-by-step explanations
    
    Always show your work and explain your reasoning clearly.
    Use the calculate tool for any mathematical computations.
    """,
    tools=[calculate]
)

# Set up handoffs for the main agent
from agents.realtime import realtime_handoff

main_agent.handoffs = [
    realtime_handoff(weather_agent, tool_description="Transfer to weather specialist for detailed weather information"),
    realtime_handoff(math_agent, tool_description="Transfer to math helper for calculations and math questions")
]

print("🤖 GPT-Realtime agents created successfully!")
print("   - Main Agent: General conversation with tools")
print("   - Weather Specialist: Detailed weather information")
print("   - Math Helper: Calculations and mathematical assistance")

In [ ]:
# Application state management for Gradio interface

@dataclass
class RealtimeAppState:
    """Application state for managing the realtime session"""
    session: Optional[RealtimeSession] = None
    runner: Optional[RealtimeRunner] = None
    is_connected: bool = False
    conversation_history: list = None
    audio_queue: list = None
    
    def __post_init__(self):
        if self.conversation_history is None:
            self.conversation_history = []
        if self.audio_queue is None:
            self.audio_queue = []

# Global state instance
app_state = RealtimeAppState()

async def create_realtime_session() -> Tuple[RealtimeRunner, RealtimeSession]:
    """Create and configure a new realtime session"""
    
    # Configure the realtime runner with GPT-Realtime model
    runner = RealtimeRunner(
        starting_agent=main_agent,
        config={
            "model_settings": {
                "model_name": "gpt-realtime",  # Use the new gpt-realtime model
                "voice": "marin",  # One of the new voices (marin or cedar)
                "modalities": ["text", "audio"],
                "input_audio_transcription": {
                    "model": "whisper-1"
                },
                "turn_detection": {
                    "type": "server_vad",  # Server-side voice activity detection
                    "threshold": 0.5,
                    "prefix_padding_ms": 300,
                    "silence_duration_ms": 500  # Wait 500ms of silence before processing
                },
                "input_audio_format": "pcm16",
                "output_audio_format": "pcm16"
            },
            "guardrails_settings": {
                "input": True,
                "output": True
            }
        }
    )
    
    # Start the session
    session = await runner.run()
    
    return runner, session

print("🔧 Realtime session configuration ready!")
print("   - Model: gpt-realtime (latest speech-to-speech model)")
print("   - Voice: marin (new enhanced voice)")
print("   - Features: Voice activity detection, transcription, guardrails")

In [ ]:
# Core realtime session management functions

async def connect_session():
    """Connect to the OpenAI Realtime API"""
    try:
        if app_state.is_connected:
            return "Already connected!"
        
        print("🔌 Connecting to OpenAI Realtime API...")
        app_state.runner, app_state.session = await create_realtime_session()
        app_state.is_connected = True
        app_state.conversation_history = []
        
        print("✅ Successfully connected to GPT-Realtime!")
        return "🎙️ Connected to GPT-Realtime! You can now start speaking."
        
    except Exception as e:
        print(f"❌ Connection failed: {str(e)}")
        app_state.is_connected = False
        return f"❌ Connection failed: {str(e)}"

async def disconnect_session():
    """Disconnect from the OpenAI Realtime API"""
    try:
        if app_state.session:
            await app_state.session.__aexit__(None, None, None)
        
        app_state.session = None
        app_state.runner = None
        app_state.is_connected = False
        
        print("🔌 Disconnected from GPT-Realtime")
        return "🔌 Disconnected from GPT-Realtime"
        
    except Exception as e:
        print(f"⚠️ Disconnect error: {str(e)}")
        return f"⚠️ Disconnect error: {str(e)}"

async def send_audio_to_realtime(audio_data: np.ndarray):
    """Send audio data to the realtime session"""
    if not app_state.session or not app_state.is_connected:
        return "❌ Not connected to realtime session"
    
    try:
        # Convert numpy array to bytes
        if audio_data.dtype != np.int16:
            audio_data = (audio_data * 32767).astype(np.int16)
        
        audio_bytes = audio_data.tobytes()
        
        # Send audio to the realtime session
        await app_state.session.send_audio(audio_bytes)
        
        return "🎤 Audio sent to GPT-Realtime"
        
    except Exception as e:
        print(f"❌ Audio send error: {str(e)}")
        return f"❌ Audio send error: {str(e)}"

async def send_text_to_realtime(message: str):
    """Send a text message to the realtime session"""
    if not app_state.session or not app_state.is_connected:
        return "❌ Not connected to realtime session"
    
    try:
        await app_state.session.send_message(message)
        app_state.conversation_history.append({"role": "user", "content": message})
        return f"📤 Message sent: {message}"
        
    except Exception as e:
        print(f"❌ Message send error: {str(e)}")
        return f"❌ Message send error: {str(e)}"

print("⚡ Realtime session management functions ready!")

In [ ]:
# Event processing for realtime session

async def process_realtime_events():
    """Process events from the realtime session"""
    if not app_state.session or not app_state.is_connected:
        return
    
    try:
        async for event in app_state.session:
            event_type = event.type
            
            print(f"📥 Received event: {event_type}")
            
            # Handle different event types
            if event_type == "response.audio_transcript.done":
                # Assistant finished speaking
                transcript = event.transcript
                app_state.conversation_history.append({
                    "role": "assistant", 
                    "content": transcript
                })
                print(f"🤖 Assistant: {transcript}")
                
            elif event_type == "conversation.item.input_audio_transcription.completed":
                # User speech transcribed
                transcript = event.transcript
                app_state.conversation_history.append({
                    "role": "user", 
                    "content": transcript
                })
                print(f"👤 User: {transcript}")
                
            elif event_type == "response.audio.delta":
                # Streaming audio from assistant
                if hasattr(event, 'delta') and event.delta:
                    # Store audio chunks for playback
                    app_state.audio_queue.append(event.delta)
                    
            elif event_type == "response.audio_transcript.delta":
                # Streaming transcript from assistant
                if hasattr(event, 'delta'):
                    print(f"🔄 Assistant (streaming): {event.delta}", end="", flush=True)
                    
            elif event_type == "input_audio_buffer.speech_started":
                print("🎤 User started speaking")
                
            elif event_type == "input_audio_buffer.speech_stopped":
                print("🎤 User stopped speaking")
                
            elif event_type == "response.function_call_arguments.delta":
                # Function call in progress
                print(f"🔧 Function call: {event.name if hasattr(event, 'name') else 'unknown'}")
                
            elif event_type == "error":
                print(f"❌ Error: {event.error}")
                break
                
    except Exception as e:
        print(f"❌ Event processing error: {str(e)}")

def get_conversation_history():
    """Get the current conversation history for display"""
    if not app_state.conversation_history:
        return "No conversation yet. Click Connect and start talking!"
    
    history_text = ""
    for i, message in enumerate(app_state.conversation_history[-10:]):  # Show last 10 messages
        role = "👤 You" if message["role"] == "user" else "🤖 Assistant"
        history_text += f"{role}: {message['content']}\n\n"
    
    return history_text.strip()

print("📊 Event processing system ready!")
print("   - Real-time transcription display")
print("   - Audio streaming handling")
print("   - Function call monitoring")
print("   - Error handling and recovery")

In [ ]:
# Gradio interface functions

def connect_button_click():
    """Handle connect button click"""
    try:
        # Run async connection in sync context
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        result = loop.run_until_complete(connect_session())
        loop.close()
        
        # Start event processing in background
        if app_state.is_connected:
            import threading
            def run_event_processor():
                loop = asyncio.new_event_loop()
                asyncio.set_event_loop(loop)
                loop.run_until_complete(process_realtime_events())
                loop.close()
            
            thread = threading.Thread(target=run_event_processor, daemon=True)
            thread.start()
        
        return result, gr.Button(interactive=False), gr.Button(interactive=True)
    except Exception as e:
        return f"❌ Connection error: {str(e)}", gr.Button(interactive=True), gr.Button(interactive=False)

def disconnect_button_click():
    """Handle disconnect button click"""
    try:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        result = loop.run_until_complete(disconnect_session())
        loop.close()
        
        return result, gr.Button(interactive=True), gr.Button(interactive=False)
    except Exception as e:
        return f"❌ Disconnect error: {str(e)}", gr.Button(interactive=True), gr.Button(interactive=False)

def send_text_message(message):
    """Handle text message sending"""
    if not message.strip():
        return "Please enter a message", ""
    
    try:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        result = loop.run_until_complete(send_text_to_realtime(message))
        loop.close()
        
        return result, ""
    except Exception as e:
        return f"❌ Send error: {str(e)}", message

def process_audio_input(audio):
    """Handle audio input from Gradio"""
    if audio is None:
        return "No audio received"
    
    try:
        # Gradio returns audio as (sample_rate, numpy_array) tuple
        if isinstance(audio, tuple):
            sample_rate, audio_data = audio
        else:
            sample_rate = SAMPLE_RATE
            audio_data = audio
        
        # Ensure correct format
        if sample_rate != SAMPLE_RATE:
            # Resample if needed (basic resampling)
            import scipy.signal
            num_samples = int(len(audio_data) * SAMPLE_RATE / sample_rate)
            audio_data = scipy.signal.resample(audio_data, num_samples)
        
        # Convert to int16 if needed
        if audio_data.dtype != np.int16:
            if audio_data.dtype == np.float32 or audio_data.dtype == np.float64:
                audio_data = (audio_data * 32767).astype(np.int16)
            else:
                audio_data = audio_data.astype(np.int16)
        
        # Send audio asynchronously
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        result = loop.run_until_complete(send_audio_to_realtime(audio_data))
        loop.close()
        
        return result
    except Exception as e:
        return f"❌ Audio processing error: {str(e)}"

def refresh_conversation():
    """Refresh the conversation display"""
    return get_conversation_history()

print("🎨 Gradio interface functions ready!")
print("   - Connect/disconnect session management")
print("   - Text message sending")
print("   - Audio input processing")
print("   - Conversation history display")

In [ ]:
# Create the Gradio interface

def create_gradio_interface():
    """Create and configure the Gradio interface"""
    
    # Custom CSS for better styling
    css = """
    .main-container {
        max-width: 1200px;
        margin: 0 auto;
        padding: 20px;
    }
    
    .status-box {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        color: white;
        padding: 15px;
        border-radius: 10px;
        text-align: center;
        font-weight: bold;
        margin: 10px 0;
    }
    
    .conversation-box {
        background: #f8f9fa;
        border: 1px solid #dee2e6;
        border-radius: 8px;
        padding: 15px;
        max-height: 400px;
        overflow-y: auto;
        white-space: pre-wrap;
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
        line-height: 1.5;
    }
    
    .audio-container {
        border: 2px dashed #007acc;
        border-radius: 10px;
        padding: 20px;
        text-align: center;
        background: #f0f8ff;
    }
    """
    
    with gr.Blocks(css=css, title="GPT-Realtime Demo") as interface:
        
        # Header
        gr.Markdown("""
        # 🎙️ OpenAI GPT-Realtime Demo
        
        **Experience the latest GPT-Realtime model** - OpenAI's most advanced speech-to-speech AI with:
        - 🗣️ Native voice conversation (no text intermediary)
        - ⚡ Ultra-low latency responses  
        - 🎯 Enhanced instruction following
        - 🔧 Advanced function calling
        - 🌟 New Marin voice with improved naturalness
        
        ---
        """)
        
        # Connection status and controls
        with gr.Row():
            with gr.Column(scale=2):
                status_display = gr.Textbox(
                    label="🔌 Connection Status",
                    value="Not connected - Click Connect to start",
                    interactive=False,
                    elem_classes=["status-box"]
                )
            
            with gr.Column(scale=1):
                connect_btn = gr.Button(
                    "🔌 Connect", 
                    variant="primary",
                    interactive=True
                )
                disconnect_btn = gr.Button(
                    "🔌 Disconnect", 
                    variant="secondary",
                    interactive=False
                )
        
        # Main interaction area
        with gr.Row():
            # Left column: Voice interaction
            with gr.Column(scale=1):
                gr.Markdown("### 🎤 Voice Interaction")
                
                audio_input = gr.Audio(
                    label="Speak to GPT-Realtime",
                    type="numpy",
                    sources=["microphone"],
                    elem_classes=["audio-container"]
                )
                
                audio_status = gr.Textbox(
                    label="Audio Status",
                    value="Ready for audio input",
                    interactive=False
                )
                
                # Text fallback
                gr.Markdown("### ✏️ Text Backup")
                with gr.Row():
                    text_input = gr.Textbox(
                        label="Type a message",
                        placeholder="Enter your message here...",
                        lines=2
                    )
                    send_text_btn = gr.Button("📤 Send Text")
                
                text_status = gr.Textbox(
                    label="Message Status",
                    value="Ready for text input",
                    interactive=False
                )
            
            # Right column: Conversation history
            with gr.Column(scale=1):
                gr.Markdown("### 💬 Conversation History")
                
                conversation_display = gr.Textbox(
                    label="Live Conversation",
                    value="No conversation yet. Connect and start talking!",
                    lines=15,
                    max_lines=20,
                    interactive=False,
                    elem_classes=["conversation-box"]
                )
                
                refresh_btn = gr.Button("🔄 Refresh Conversation")
        
        # Instructions and examples
        with gr.Row():
            gr.Markdown("""
            ### 🎯 Try These Examples:
            
            **Voice Commands:**
            - "What's the weather in Tokyo?"
            - "What time is it?"
            - "Calculate 15 times 7"
            - "Tell me a joke"
            - "Help me with math problems"
            
            **Agent Handoffs:**
            - "I need detailed weather information" → Weather Specialist
            - "Help me with calculations" → Math Helper
            
            ### 🔧 Features Demonstrated:
            - **Real-time speech-to-speech** with GPT-Realtime
            - **Function calling** (weather, time, calculations)
            - **Agent handoffs** between specialized assistants
            - **Enhanced voice quality** with the new Marin voice
            - **Improved instruction following** and natural conversation
            """)
        
        # Event handlers
        connect_btn.click(
            connect_button_click,
            outputs=[status_display, connect_btn, disconnect_btn]
        )
        
        disconnect_btn.click(
            disconnect_button_click,
            outputs=[status_display, connect_btn, disconnect_btn]
        )
        
        audio_input.change(
            process_audio_input,
            inputs=[audio_input],
            outputs=[audio_status]
        )
        
        send_text_btn.click(
            send_text_message,
            inputs=[text_input],
            outputs=[text_status, text_input]
        )
        
        refresh_btn.click(
            refresh_conversation,
            outputs=[conversation_display]
        )
        
        # Auto-refresh conversation every 2 seconds when connected
        interface.load(
            refresh_conversation,
            outputs=[conversation_display],
            every=2
        )
    
    return interface

print("🎨 Gradio interface created successfully!")
print("   - Modern, responsive design")
print("   - Voice and text input options")
print("   - Real-time conversation display")
print("   - Connection status monitoring")
print("   - Example commands and instructions")

In [ ]:
# Launch the GPT-Realtime demo

def launch_demo():
    """Launch the GPT-Realtime demo interface"""
    
    print("🚀 Launching GPT-Realtime Demo...")
    print("="*60)
    print("🎙️  OpenAI GPT-Realtime Speech-to-Speech Demo")
    print("📡  Model: gpt-realtime (Latest)")
    print("🗣️  Voice: Marin (Enhanced Natural Speech)")
    print("⚡  Features: Real-time conversation, function calling, agent handoffs")
    print("="*60)
    
    # Create and launch the interface
    interface = create_gradio_interface()
    
    # Launch with public sharing enabled
    interface.launch(
        share=True,  # Create public URL for sharing
        server_name="0.0.0.0",  # Allow external connections
        server_port=7860,  # Standard Gradio port
        show_api=True,  # Show API documentation
        debug=True,  # Enable debug mode
        max_threads=10,  # Handle multiple concurrent users
        inbrowser=True  # Open browser automatically
    )

# Run the demo
if __name__ == "__main__":
    # Check prerequisites
    if not openai_api_key:
        print("❌ Missing OpenAI API key. Please set OPENAI_API_KEY in your .env file")
    else:
        launch_demo()
else:
    print("✅ Demo ready to launch!")
    print("💡 Run launch_demo() to start the GPT-Realtime interface")

In [ ]:
# Quick test of GPT-Realtime connection (optional)

async def test_gpt_realtime_connection():
    """Test the GPT-Realtime connection without the full interface"""
    
    print("🧪 Testing GPT-Realtime connection...")
    
    try:
        # Create a test agent
        test_agent = RealtimeAgent(
            name="Test Assistant",
            instructions="You are a test assistant. Respond briefly to confirm the connection is working."
        )
        
        # Create runner
        runner = RealtimeRunner(
            starting_agent=test_agent,
            config={
                "model_settings": {
                    "model_name": "gpt-realtime",
                    "voice": "marin",
                    "modalities": ["text", "audio"]
                }
            }
        )
        
        # Test session
        session = await runner.run()
        
        async with session:
            print("✅ Connected to GPT-Realtime successfully!")
            
            # Send a test message
            await session.send_message("Hello, can you confirm the connection is working?")
            
            # Wait for a few events
            event_count = 0
            async for event in session:
                print(f"📥 Event: {event.type}")
                
                if event.type == "response.audio_transcript.done":
                    print(f"🤖 Response: {event.transcript}")
                    break
                    
                event_count += 1
                if event_count > 10:  # Prevent infinite loop
                    break
        
        print("✅ Test completed successfully!")
        return True
        
    except Exception as e:
        print(f"❌ Test failed: {str(e)}")
        return False

def run_connection_test():
    """Run the connection test in a sync context"""
    try:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        result = loop.run_until_complete(test_gpt_realtime_connection())
        loop.close()
        return result
    except Exception as e:
        print(f"❌ Test runner error: {str(e)}")
        return False

print("🧪 Connection test function ready!")
print("💡 Run run_connection_test() to verify GPT-Realtime API access")

## 🚀 Ready to Launch!

Your **GPT-Realtime Demo** is now ready! This implementation showcases:

### 🎯 Key Features Implemented:

1. **🗣️ Native Speech-to-Speech**: Direct audio conversation with the new `gpt-realtime` model
2. **⚡ Ultra-Low Latency**: Real-time responses with minimal delay
3. **🎵 Enhanced Voice Quality**: Using the new "Marin" voice with improved naturalness
4. **🔧 Function Calling**: Weather, time, and calculation tools
5. **🤝 Agent Handoffs**: Specialized weather and math assistants
6. **🌐 Web Interface**: Beautiful Gradio interface with voice and text input
7. **📊 Real-time Monitoring**: Live conversation history and status updates

### 🚀 How to Launch:

```python
# Option 1: Launch the full demo
launch_demo()

# Option 2: Test connection first
run_connection_test()
```

### 💡 Usage Instructions:

1. **Connect**: Click the "Connect" button to establish a session with GPT-Realtime
2. **Speak**: Use the microphone to have voice conversations
3. **Text**: Use the text input as a backup option
4. **Monitor**: Watch the live conversation history update in real-time
5. **Functions**: Try commands like "What's the weather in Tokyo?" or "Calculate 15 times 7"
6. **Handoffs**: Request specialized help to see agent handoffs in action

### 🔧 Technical Details:

- **Model**: `gpt-realtime` (OpenAI's latest speech-to-speech model)
- **Voice**: "Marin" (one of the new enhanced voices)
- **Audio Format**: PCM16, 24kHz, Mono
- **Framework**: OpenAI Agents SDK with Gradio UI
- **Features**: Voice activity detection, transcription, guardrails

This implementation demonstrates the cutting-edge capabilities of OpenAI's GPT-Realtime model and how to integrate it into web applications using the Agents SDK and Gradio!